# CRISPR Guide AI: Notebook 04 - Rigorous Model Evaluation & Explainability

### Objective:
Analyze model performance across multiple evaluation dimensions:
1. Full metrics table (MAE, RMSE, R², Pearson, Spearman, ROC-AUC, PR-AUC)
2. Investigation of data leakage: Random Split vs. Cross-Cell-Line (Leave-One-Cell-Out)
3. Residual analysis and error homoscedasticity
4. Global and local feature explainability (SHAP / Gini feature importances and CNN positional saliency)


In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

SAVED_DIR = os.path.join("..", "backend", "saved_models")
with open(os.path.join(SAVED_DIR, "metrics.json")) as f:
    metrics = json.load(f)

# Random Split comparison table
random_df = pd.DataFrame(metrics["random_split"]).T
random_df = random_df[["pearson", "spearman", "rmse", "mae", "r2", "roc_auc", "pr_auc"]]
random_df.columns = ["Pearson r", "Spearman rho", "RMSE", "MAE", "R2", "ROC-AUC", "PR-AUC"]
print("Random 75/25 Train-Test Split Results:")
random_df


In [ ]:
# Leave-One-Cell-Out Cross-Validation Table
loco_data = {}
for cell, val in metrics.get("leave_cell_out", {}).items():
    m = val["metrics"]
    loco_data[f"Held-Out: {cell}"] = {
        "Samples": val["held_out_samples"],
        "Pearson r": m["pearson"],
        "Spearman rho": m["spearman"],
        "RMSE": m["rmse"],
        "ROC-AUC": m["roc_auc"]
    }
loco_df = pd.DataFrame(loco_data).T
print("Leave-One-Cell-Out Generalization Performance:")
loco_df


In [ ]:
# Top 15 Predictive Sequence Features Across the Genome
with open(os.path.join(SAVED_DIR, "feature_importance.json")) as f:
    feat_imp = json.load(f)

imp_df = pd.DataFrame(feat_imp).head(15)
plt.figure(figsize=(10, 6))
plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1], color="#2563eb")
plt.title("Top 15 Most Predictive Sequence Features (XGBoost Importance)")
plt.xlabel("Relative Feature Importance")
plt.tight_layout()
plt.show()
